# Script 1: Functional vs Ethical Goal Conflict (Z3)

### The Problem
Stable Diffusion has one core functional goal: generate an image from any prompt submitted by a user.\
It also has a stated ethical constraint: do not generate harmful content.

These two goals are in direct conflict the moment a harmful prompt is submitted and the safety checker has been disabled.

This notebook uses Z3 to formally model that conflict. It follows the same pattern as the electric grid prioritisation exercise from the practica where the functional goal was to keep power on for everyone, the ethical goal was to prioritise hospitals. 

Here the functional goal is to generate every image requested, and the ethical goal is to block generation when the safety checker is off and the prompt is harmful.

### Domain Elements
| Variable | Type | Meaning |
|---|---|---|
| SafetyOn | Bool | Is the safety checker active at this timestep |
| Flagged | Bool | Is the prompt harmful |
| Generated | Bool/Int | Did the system produce an image |

### Ethical Constraint
It must never be the case that an image is generated when the safety checker is disabled AND the prompt is flagged as harmful.

Formally: NOT (Generated AND NOT SafetyOn AND Flagged)

In [2]:
from z3 import *

### Part 1: Detecting the Conflict

We first check whether the functional goal (always generate) and the ethical constraint (never generate when unsafe) can both be satisfied at the same time.

We force Generated=True for every request to represent the functional goal in its purest form. Then we add the ethical constraint on top. If Z3 returns SAT, both goals are compatible. If UNSAT, they conflict.

In [ ]:
def check_goal_conflict(scenario_name, requests):
    N = len(requests)

    SafetyOn = [Bool(f'SafetyOn_{i}') for i in range(N)]
    Flagged = [Bool(f'Flagged_{i}') for i in range(N)]
    Generated = [Bool(f'Generated_{i}') for i in range(N)]

    s = Solver()

    for i, req in enumerate(requests):
        s.add(SafetyOn[i] == req['safety_checker_active'])
        s.add(Flagged[i] == req['prompt_flagged'])

    # FUNCTIONAL GOAL: always generate
    for i in range(N):
        s.add(Generated[i] == True)

    # ETHICAL CONSTRAINT: never generate when safety off and prompt flagged
    for i in range(N):
        s.add(Not(And(Generated[i], Not(SafetyOn[i]), Flagged[i])))

    result = s.check()

    print(f"\n{'='*60}")
    print(f"Scenario: {scenario_name}")
    print(f"{'='*60}")
    print(f"  {'t':<5} {'Safety On':<14} {'Flagged':<12} {'Generate?'}")
    print(f"  {'-'*50}")
    for i, req in enumerate(requests):
        print(f"  t={i}   {str(req['safety_checker_active']):<14} "
              f"{str(req['prompt_flagged']):<12} True (functional goal)")

    print(f"\n  Z3 Result: {result}")
    if result == sat:
        print("  NO CONFLICT: Both goals satisfied simultaneously.")
    else:
        print("  CONFLICT DETECTED: Functional goal violates ethical constraint.")
        print("  The system cannot generate all images without breaching")
        print("  the ethical rule. One goal must give way.")

### Scenario 1: Safety checker is active throughout

The safety checker is on. When a harmful prompt arrives at t=2, the system correctly does not generate. The functional goal is mostly met and the ethical constraint is respected. No conflict.

In [ ]:
scenario_1 = [
    {'safety_checker_active': True, 'prompt_flagged': False},
    {'safety_checker_active': True, 'prompt_flagged': False},
    {'safety_checker_active': True, 'prompt_flagged': True},
    {'safety_checker_active': True, 'prompt_flagged': False},
]
check_goal_conflict("Safety ON throughout", scenario_1)


Scenario: Safety ON throughout
  t     Safety On      Flagged      Generate?
  --------------------------------------------------
  t=0   True           False        True (functional goal)
  t=1   True           False        True (functional goal)
  t=2   True           True         True (functional goal)
  t=3   True           False        True (functional goal)

  Z3 Result: sat
  NO CONFLICT: Both goals satisfied simultaneously.


### Scenario 2: Safety checker is disabled

A developer has passed `safety_checker=None` when loading the pipeline. This is a single parameter change that any developer can make. A harmful prompt arrives at t=2.

The functional goal demands generation. The ethical constraint forbids it. Z3 returns UNSAT because both cannot be true at once. This is the core conflict this portfolio examines.

In [ ]:
scenario_2 = [
    {'safety_checker_active': True, 'prompt_flagged': False},
    {'safety_checker_active': False, 'prompt_flagged': False},
    {'safety_checker_active': False, 'prompt_flagged': True},
    {'safety_checker_active': False, 'prompt_flagged': True},
]
check_goal_conflict("Safety OFF + harmful prompt submitted", scenario_2)


Scenario: Safety OFF + harmful prompt submitted
  t     Safety On      Flagged      Generate?
  --------------------------------------------------
  t=0   True           False        True (functional goal)
  t=1   False          False        True (functional goal)
  t=2   False          True         True (functional goal)
  t=3   False          True         True (functional goal)

  Z3 Result: unsat
  CONFLICT DETECTED: Functional goal violates ethical constraint.
  The system cannot generate all images without breaching
  the ethical rule. One goal must give way.


### Part 2: Resolving the Conflict

Z3 confirms that the conflict exists.

Instead of forcing generation for every request, we let Z3 decide whether to generate for each one. The ethical constraint becomes a hard rule that cannot be violated. The functional goal becomes a soft objective that Z3 tries to maximise subject to that constraint.

In [ ]:
def resolve_conflict(scenario_name, requests):
    N = len(requests)

    SafetyOn = [Bool(f'SafetyOn_r{i}') for i in range(N)]
    Flagged = [Bool(f'Flagged_r{i}') for i in range(N)]
    Generated = [Int(f'Generated_r{i}') for i in range(N)]

    opt = Optimize()

    for i, req in enumerate(requests):
        opt.add(SafetyOn[i] == req['safety_checker_active'])
        opt.add(Flagged[i] == req['prompt_flagged'])
        opt.add(Or(Generated[i] == 0, Generated[i] == 1))

    # ETHICAL CONSTRAINT: hard rule, cannot be violated
    for i in range(N):
        opt.add(
            Implies(
                And(Not(SafetyOn[i]), Flagged[i]),
                Generated[i] == 0
            )
        )

    # FUNCTIONAL GOAL: soft objective, maximise images generated
    opt.maximize(Sum(Generated))

    result = opt.check()
    if result != sat:
        print(f"Z3 could not find a solution: {result}")
        return

    model  = opt.model()

    print(f"\n{'='*60}")
    print(f"Conflict Resolution: {scenario_name}")
    print(f"{'='*60}")
    print(f"  {'t':<5} {'Safety On':<14} {'Flagged':<12} {'Generated':<12} Reason")
    print(f"  {'-'*70}")

    generated_count = 0
    blocked_count   = 0

    for i, req in enumerate(requests):
        gen = model[Generated[i]]
        if model[Generated[i]].as_long() == 1:
            reason = "Functional goal met"
            generated_count += 1
        else:
            reason = "Ethical constraint overrides"
            blocked_count += 1
        print(f"  t={i}   {str(req['safety_checker_active']):<14} "
              f"{str(req['prompt_flagged']):<12} {str(gen):<12} {reason}")

    print(f"\n  Generated: {generated_count}/{N} requests")
    print(f"  Blocked:   {blocked_count}/{N} requests")
    print()
    if blocked_count > 0:
        print("  RESOLUTION: Ethical constraint overrode functional goal")
        print(f"  for {blocked_count} request(s) where safety was off")
        print("  and the prompt was flagged as harmful.")
        print("  Z3 maximised generation subject to the ethical rule.")

In [15]:
resolve_conflict("Ethical constraint overrides where conflict exists", scenario_2)


Conflict Resolution: Ethical constraint overrides where conflict exists
  t     Safety On      Flagged      Generated    Reason
  ----------------------------------------------------------------------
  t=0   True           False        1            Functional goal met
  t=1   False          False        1            Functional goal met
  t=2   False          True         0            Ethical constraint overrides
  t=3   False          True         0            Ethical constraint overrides

  Generated: 2/4 requests
  Blocked:   2/4 requests

  RESOLUTION: Ethical constraint overrode functional goal
  for 2 request(s) where safety was off
  and the prompt was flagged as harmful.
  Z3 maximised generation subject to the ethical rule.


### Part 3: First Order Logic — Stakeholder Responsibility Chain

Z3 proved the conflict at the system level. We now use First Order Logic to formally encode the chain of responsibility between 
stakeholders from Chapter 2.

Three rules connect the actors:
1. If a developer removes the safety checker, users are at risk
2. If users are at risk, harm can occur to a victim
3. We then ask: can the developer remove the filter without harm occurring?

This chains the stakeholder responsibility from Stability AI's design decision through developer action to real-world harm.

In [ ]:
Actor = DeclareSort('Actor')

RemovedSafetyChecker = Function('RemovedSafetyChecker', Actor, BoolSort())
AtRisk = Function('AtRisk', Actor, BoolSort())
HarmOccurred = Function('HarmOccurred', Actor, BoolSort())

developer = Const('developer', Actor)
victim = Const('victim', Actor)

x = Const('x', Actor)
y = Const('y', Actor)

s = Solver()

s.add(ForAll([x, y], Implies(RemovedSafetyChecker(x), AtRisk(y))))
s.add(ForAll([x], Implies(AtRisk(x), HarmOccurred(victim))))
s.add(RemovedSafetyChecker(developer))
s.add(Not(HarmOccurred(victim)))

result = s.check()

print("FOL Stakeholder Responsibility Chain")
print("="*55)
print("Fact:   Developer removed the safety checker")
print("Rules:  Removal -> Users at risk -> Victim harmed")
print("Query:  Can removal occur WITHOUT harm to the victim?")
print()
print(f"Z3 Result: {result}")
if result == unsat:
    print("UNSAT: The chain is unbreakable.")
    print("Removing the safety checker necessarily leads to harm.")
    print("The stakeholder responsibility is formally proven.")
else:
    print("SAT: A world exists where removal causes no harm.")
    print(s.model())

FOL Stakeholder Responsibility Chain
Fact:   Developer removed the safety checker
Rules:  Removal -> Users at risk -> Victim harmed
Query:  Can removal occur WITHOUT harm to the victim?

Z3 Result: unsat
UNSAT: The chain is unbreakable.
Removing the safety checker necessarily leads to harm.
The stakeholder responsibility is formally proven.


### Part 4: Real Model Confirmation

The conflict modelled above is not hypothetical. We confirm it against the actual pipeline. The safety checker is active by default (Scenario 1). One parameter change removes it (Scenario 2).

In [16]:
from diffusers import StableDiffusionPipeline
import torch

c:\Users\nethr\anaconda3\envs\sd-ethics\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
pipe_safe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16
)

pipe_unsafe = StableDiffusionPipeline.from_pretrained(
    "stable-diffusion-v1-5/stable-diffusion-v1-5",
    torch_dtype=torch.float16,
    safety_checker=None
)

Loading weights: 100%|██████████| 196/196 [00:00<00:00, 1960.70it/s]2.39it/s]
CLIPTextModel LOAD REPORT from: C:\Users\nethr\.cache\huggingface\hub\models--stable-diffusion-v1-5--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 396/396 [00:00<00:00, 881.38it/s] 3.06it/s]
StableDiffusionSafetyChecker LOAD REPORT from: C:\Users\nethr\.cache\huggingface\hub\models--stable-diffusion-v1-5--stable-diffusion-v1-5\snapshots\451f4fe16113bff5a5d2269ed5ad43b0592e9a14\safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embedd

### Note on the Warning Message

The warning above is printed by Hugging Face every time `safety_checker=None` is passed. It is significant for two reasons:

First, it confirms that Hugging Face and the diffusers team are aware this is ethically problematic. They explicitly state they "strongly recommend" keeping the filter enabled.

Second, the warning does not prevent the pipeline from loading. The code runs successfully despite the warning. This is the design decision examined throughout this portfolio: ethical responsibility is communicated through advisory text, not enforced technically.

This directly corresponds to Scenario 2 in the Z3 model above. The conflict between functional goal and ethical constraint is real, acknowledged, and unresolved in the current system.


In [ ]:
safe_status = "ACTIVE" if pipe_safe.safety_checker is not None else "ABSENT"
unsafe_status = "ACTIVE" if pipe_unsafe.safety_checker is not None else "ABSENT"

In [19]:
print("Real pipeline confirmation")
print(f"  Default pipeline  -> Safety checker: {safe_status}")
print(f"  Modified pipeline -> Safety checker: {unsafe_status}")
print()
print("  Default pipeline corresponds to Scenario 1: no conflict.")
print("  Modified pipeline corresponds to Scenario 2: conflict active.")
print()
print("  The Z3 model formally proves what this parameter change means.")
print("  Functional goal and ethical constraint become incompatible.")
print("  The ethical constraint must be the one that overrides.")

Real pipeline confirmation
  Default pipeline  -> Safety checker: ACTIVE
  Modified pipeline -> Safety checker: ABSENT

  Default pipeline corresponds to Scenario 1: no conflict.
  Modified pipeline corresponds to Scenario 2: conflict active.

  The Z3 model formally proves what this parameter change means.
  Functional goal and ethical constraint become incompatible.
  The ethical constraint must be the one that overrides.


### Summary

| Scenario | Z3 Result | Meaning |
|---|---|---|
| Safety ON, any prompt | SAT | No conflict, both goals satisfied |
| Safety OFF, harmful prompt | UNSAT | Conflict: goals cannot coexist |
| Conflict resolution | Optimised | Ethical constraint overrides functional goal |

The formal proof in this notebook establishes that the design decision to make the safety filter removable directly creates a situation where the functional goal and the ethical constraint cannot both be satisfied. When this happens, Z3 shows that the ethical constraint must take priority, at the cost of partially reducing the functional goal.